<a href="https://colab.research.google.com/github/adapasriram/Projectss/blob/main/Copy_of_Untitled75.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
df = pd.read_csv('Hindi_English_Truncated_Corpus.csv (1).zip')
df

In [ ]:
print(df.columns.tolist())

In [ ]:
df = df[
    df['source'] == 'ted'
][['english_sentence','hindi_sentence']].copy()




In [ ]:
df

In [ ]:
df = df.dropna()

df['english_sentence'] = (
    df['english_sentence'].astype(str).str.strip()
)

df['hindi_sentence'] = (
    df['hindi_sentence'].astype(str).str.strip()
)

df = df[
    (df['english_sentence'] != "") &
    (df['hindi_sentence'] != "")
]

df = df.drop_duplicates().reset_index(drop=True)

print(df.shape)

In [ ]:
## Text preprocessing

import re

def clean_english(text):

  text = text.lower()

  ## remove punctuation
  text = re.sub(r'[^\w\s]',"",text)

  ## remove digits
  text = re.sub(r'\d+',"",text)

  ## remove extra spaces
  text = re.sub(r'\s+'," ",text)

  return text.strip()

def clean_hindi(text):
  text = re.sub(r"\s+"," ",text)

  return text.strip()

In [ ]:
df['english_sentence'] = (
    df['english_sentence'].apply(clean_english)
)

df['hindi_sentence'] = (
    df['hindi_sentence'].apply(clean_hindi)
)

df.head()

In [ ]:
## Select training samples

MAX_LEN = 30

df = df[
    (df['english_sentence'].str.split().str.len() <= MAX_LEN - 1) &
    (df['hindi_sentence'].str.split().str.len() <= MAX_LEN - 2)
].copy()

df = df.sample(
    n = min(20000,len(df)),random_state=42
).reset_index(drop=True)

print(len(df))

In [ ]:
## apply sos and eos in my hindi sentence
df['hindi_sentence'] = df['hindi_sentence'].apply(
    lambda x: '<sos>' + x + '<eos>'
)

df[['english_sentence','hindi_sentence']].head()

In [ ]:
## Train test split

from sklearn.model_selection import train_test_split
train_df , test_df = train_test_split(
    df,test_size=0.2,random_state=42
)

print('Training sample',len(train_df))
print('Testing sample',len(test_df))


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
## English Tokenizer
NUM_WORDS = 1500

eng_tokenizer = Tokenizer(
    num_words = NUM_WORDS,
    oov_token = "<unk"
)

eng_tokenizer.fit_on_texts(
    train_df['english_sentence']
)

eng_train_seq = eng_tokenizer.texts_to_sequences(
    train_df['english_sentence']
)

eng_test_seq = eng_tokenizer.texts_to_sequences(
    test_df['english_sentence']
)

In [ ]:
## Hindi Tokenizer

## English Tokenizer
NUM_WORDS = 1500

hin_tokenizer = Tokenizer(
    num_words = NUM_WORDS,
    filters ="",
    oov_token = "<unk"
)

hin_tokenizer.fit_on_texts(
    train_df['hindi_sentence']
)

hin_train_seq = hin_tokenizer.texts_to_sequences(
    train_df['hindi_sentence']
)

hin_test_seq = hin_tokenizer.texts_to_sequences(
    test_df['hindi_sentence']
)

In [ ]:
## Calaculate the vocabulary size

eng_vocab_size = min(
    NUM_WORDS,
    len(eng_tokenizer.word_index) + 1
)

hin_vocab_size = min(
    NUM_WORDS,
    len(hin_tokenizer.word_index) + 1
)

print('English vocab size',eng_vocab_size)
print('Hindi vocab size',hin_vocab_size)
